# ДЗ-10

## 1. постановка задачи

матрица $A$ размера $8 \times 8$ строится из sha-256 хэша строки Бабаева А.В. 501290. шестьдесят четыре шестнадцатеричные цифры хэша записываются по строкам слева направо сверху вниз и переводятся в десятичную систему.

для матрицы $M = A A^{T}$ требуется реализовать степенной метод и найти наибольшее по модулю собственное число вместе с соответствующим собственным вектором 

матрица $M = A A^{T}$ симметрична и положительно полуопределена, поэтому её спектр вещественный и неотрицательный, что обеспечивает применимость степенного метода.

## 2. построение матрицы

In [ ]:
import numpy as np
import hashlib

In [ ]:
s = 'Бабаева А.В. 501290'
h = hashlib.sha256(s.encode('utf-8')).hexdigest()
print('sha-256:', h)

A = np.array([int(c, 16) for c in h]).reshape(8, 8).astype(float)
print('матрица A:')
print(A.astype(int))

In [ ]:
M = A @ A.T
print('матрица M = A A^T:')
print(M.astype(int))
print()
print('симметричность M:', np.allclose(M, M.T))

## 3. степенной метод

идея метода состоит в построении последовательности итерированных векторов. на каждом шаге вычисляется $y = M x$, после чего вектор нормируется к единичной длине, чтобы избежать переполнения или потери значащих цифр.

для симметричной матрицы оценкой наибольшего собственного числа на каждой итерации служит отношение рэлея $\lambda = (x, M x)$ при $\|x\| = 1$. итерации продолжаются до тех пор, пока изменение оценки $\lambda$ между соседними шагами не станет меньше заданной точности $\varepsilon$.

в качестве начального берётся вектор из единиц, приведённый к единичной длине.

In [ ]:
n = M.shape[0]
eps = 1e-9

x = np.ones(n)
x = x / np.linalg.norm(x)

lam_old = 0.0
for k in range(1000):
    y = M @ x
    lam = x @ y
    x = y / np.linalg.norm(y)
    print(f'итерация {k+1:2d}:  lambda = {lam:.6f}')
    if abs(lam - lam_old) < eps:
        break
    lam_old = lam

## 4. результат и проверка

наибольшее по модулю собственное число и соответствующий ему нормированный собственный вектор:

In [ ]:
print('наибольшее собственное число lambda_1 =', round(lam, 4))
print()
print('собственный вектор x_1:')
print(np.round(x, 4))

проверка выполняется подстановкой в определение собственной пары: вычисляется норма невязки $\| M x - \lambda x \|$. для контроля результат сравнивается со встроенным вычислением собственных чисел.

In [ ]:
nevyazka = np.linalg.norm(M @ x - lam * x)
print('норма невязки ||M x - lambda x|| =', f'{nevyazka:.2e}')
print()
kontrol = np.linalg.eigvalsh(M)
print('наибольшее собственное число по контрольному расчёту:', round(kontrol[-1], 4))

## 5. выводы

степенной метод сошёлся за небольшое число итераций и дал наибольшее по модулю собственное число матрицы $M = A A^{T}$, равное приблизительно $4498.70$. норма невязки $\| M x - \lambda x \|$ имеет порядок $10^{-5}$, что подтверждает корректность найденной собственной пары. результат совпадает с контрольным расчётом.

быстрая сходимость объясняется тем, что наибольшее собственное число матрицы значительно превосходит остальные по модулю, и скорость сходимости степенного метода определяется отношением второго по величине собственного числа к первому.